# 03 — Asynchronous Programming

Async programming is the **heart of Node.js**. This is tested in virtually every Node.js interview.

---

## Table of Contents
1. Why Async Matters
2. Callbacks & Callback Hell
3. Promises
4. async / await
5. Promise Combinators
6. Microtasks vs Macrotasks
7. Common Async Patterns
8. Error Handling in Async Code
9. Interview Questions

---
## 1. Why Async Matters

Node.js is **single-threaded**. If you do a synchronous file read that takes 100ms, your server **cannot handle any other requests** during that time.

Async programming lets Node.js:
- Start an I/O operation
- Continue processing other requests
- Come back when the I/O is done

This is why Node.js can handle **10,000+ concurrent connections** on a single thread.

In [ ]:
const fs = require('fs');

// SYNCHRONOUS (blocking) — avoid in servers!
console.time('sync');
try {
    const data = fs.readFileSync(__filename, 'utf8');
    console.log('Sync read: got', data.length, 'chars');
} catch (err) {
    console.error(err);
}
console.timeEnd('sync');
console.log('This line waits for the file read ^^');

In [ ]:
// ASYNCHRONOUS (non-blocking) — preferred!
console.log('Starting async read...');
fs.readFile(__filename, 'utf8', (err, data) => {
    if (err) {
        console.error(err);
        return;
    }
    console.log('Async read: got', data.length, 'chars');
});
console.log('This line runs IMMEDIATELY (does not wait!)');

---
## 2. Callbacks & Callback Hell

### The Node.js callback convention (error-first callbacks):
```javascript
function asyncOperation(arg, callback) {
    // callback(error, result)
    // If success: callback(null, data)
    // If error:   callback(new Error('...'), null)
}
```

This is called the **Error-First Callback Pattern** — the first argument is always the error.

In [ ]:
// Error-first callback pattern
function divideAsync(a, b, callback) {
    setTimeout(() => {
        if (b === 0) {
            callback(new Error('Division by zero'), null);
        } else {
            callback(null, a / b);
        }
    }, 100);
}

divideAsync(10, 2, (err, result) => {
    if (err) {
        console.error('Error:', err.message);
        return;
    }
    console.log('Result:', result); // 5
});

divideAsync(10, 0, (err, result) => {
    if (err) {
        console.error('Error:', err.message); // Division by zero
        return;
    }
    console.log('Result:', result);
});

In [ ]:
// CALLBACK HELL — the pyramid of doom
// This is WHY Promises and async/await were invented

function getUser(id, cb) { setTimeout(() => cb(null, { id, name: 'Alice' }), 100); }
function getOrders(userId, cb) { setTimeout(() => cb(null, [{ id: 1, total: 50 }]), 100); }
function getProduct(orderId, cb) { setTimeout(() => cb(null, { name: 'Widget' }), 100); }

// Nested callbacks = hard to read, maintain, and debug
getUser(1, (err, user) => {
    if (err) return console.error(err);
    getOrders(user.id, (err, orders) => {
        if (err) return console.error(err);
        getProduct(orders[0].id, (err, product) => {
            if (err) return console.error(err);
            console.log(`${user.name} ordered ${product.name}`);
        });
    });
});

---
## 3. Promises

A Promise represents a value that may be available **now**, **later**, or **never**.

### Promise states:
- **Pending** — initial state
- **Fulfilled** — operation completed successfully → `.then()` runs
- **Rejected** — operation failed → `.catch()` runs

A promise is **settled** once it's either fulfilled or rejected (cannot change after that).

In [ ]:
// Creating a Promise
function dividePromise(a, b) {
    return new Promise((resolve, reject) => {
        setTimeout(() => {
            if (b === 0) {
                reject(new Error('Division by zero'));
            } else {
                resolve(a / b);
            }
        }, 100);
    });
}

// Consuming a Promise
dividePromise(10, 2)
    .then(result => console.log('Result:', result))
    .catch(err => console.error('Error:', err.message));

dividePromise(10, 0)
    .then(result => console.log('Result:', result))
    .catch(err => console.error('Error:', err.message));

In [ ]:
// Promise chaining — solving callback hell!
function getUserP(id) { return Promise.resolve({ id, name: 'Alice' }); }
function getOrdersP(userId) { return Promise.resolve([{ id: 1, total: 50 }]); }
function getProductP(orderId) { return Promise.resolve({ name: 'Widget' }); }

let savedUser;

getUserP(1)
    .then(user => {
        savedUser = user;
        return getOrdersP(user.id);
    })
    .then(orders => getProductP(orders[0].id))
    .then(product => console.log(`${savedUser.name} ordered ${product.name}`))
    .catch(err => console.error('Something failed:', err.message));

In [ ]:
// Converting callbacks to Promises (promisification)
const { promisify } = require('util');
const fs = require('fs');

// Manual promisification
function readFilePromise(path) {
    return new Promise((resolve, reject) => {
        fs.readFile(path, 'utf8', (err, data) => {
            if (err) reject(err);
            else resolve(data);
        });
    });
}

// Using util.promisify (preferred)
const readFileAsync = promisify(fs.readFile);

// Even better: fs.promises (built-in since Node 10)
const fsPromises = require('fs').promises;
// or: const fsPromises = require('fs/promises');

console.log('Three ways to promisify fs.readFile demonstrated');

---
## 4. async / await

Syntactic sugar over Promises. Makes async code look and behave like synchronous code.

### Rules:
- `async` function **always returns a Promise**
- `await` can only be used **inside an async function** (or top-level in ESM)
- `await` pauses execution of the async function until the Promise settles

In [ ]:
// The callback hell example, rewritten with async/await
function getUserP(id) { return Promise.resolve({ id, name: 'Alice' }); }
function getOrdersP(userId) { return Promise.resolve([{ id: 1, total: 50 }]); }
function getProductP(orderId) { return Promise.resolve({ name: 'Widget' }); }

async function getUserOrder() {
    const user = await getUserP(1);
    const orders = await getOrdersP(user.id);
    const product = await getProductP(orders[0].id);
    return `${user.name} ordered ${product.name}`;
}

getUserOrder().then(console.log);

In [ ]:
// async functions ALWAYS return a Promise
async function greet() {
    return 'Hello';  // Equivalent to: return Promise.resolve('Hello');
}

const result = greet();
console.log('Type:', typeof result);         // object
console.log('Is Promise:', result instanceof Promise); // true
result.then(val => console.log('Value:', val));  // Hello

In [ ]:
// Error handling with async/await
async function riskyOperation() {
    throw new Error('Something went wrong');
}

// Method 1: try/catch
async function handleError1() {
    try {
        const result = await riskyOperation();
        console.log(result);
    } catch (err) {
        console.error('Caught:', err.message);
    }
}

// Method 2: .catch() on the Promise
async function handleError2() {
    const result = await riskyOperation().catch(err => {
        console.error('Caught via .catch():', err.message);
        return 'fallback value';
    });
    console.log('Result:', result);
}

handleError1();
handleError2();

---
## 5. Promise Combinators

These are essential for handling **multiple async operations**.

| Method | Resolves when | Rejects when | Use case |
|--------|-------------|-------------|----------|
| `Promise.all()` | ALL fulfill | ANY rejects | Parallel ops, all required |
| `Promise.allSettled()` | ALL settle | Never rejects | Parallel ops, want all results |
| `Promise.race()` | FIRST settles | FIRST settles | Timeout patterns |
| `Promise.any()` | FIRST fulfills | ALL reject | Fastest success wins |

In [ ]:
const delay = (ms, value) => new Promise(resolve => setTimeout(() => resolve(value), ms));
const fail = (ms, msg) => new Promise((_, reject) => setTimeout(() => reject(new Error(msg)), ms));

// Promise.all — all must succeed
async function demoAll() {
    try {
        const results = await Promise.all([
            delay(100, 'A'),
            delay(200, 'B'),
            delay(50, 'C')
        ]);
        console.log('Promise.all:', results); // ['A', 'B', 'C']
    } catch (err) {
        console.error('One failed:', err.message);
    }
}
demoAll();

In [ ]:
// Promise.allSettled — get results of all, even failures
async function demoAllSettled() {
    const results = await Promise.allSettled([
        delay(100, 'Success'),
        fail(50, 'Oops'),
        delay(200, 'Also success')
    ]);
    console.log('Promise.allSettled:');
    results.forEach((r, i) => {
        if (r.status === 'fulfilled') {
            console.log(`  [${i}] fulfilled:`, r.value);
        } else {
            console.log(`  [${i}] rejected:`, r.reason.message);
        }
    });
}
demoAllSettled();

In [ ]:
// Promise.race — first to settle wins (useful for timeouts!)
async function fetchWithTimeout(fetchPromise, timeoutMs) {
    const timeout = new Promise((_, reject) =>
        setTimeout(() => reject(new Error('Timeout!')), timeoutMs)
    );
    return Promise.race([fetchPromise, timeout]);
}

// Simulating a slow API
const slowAPI = delay(5000, 'data');

fetchWithTimeout(slowAPI, 1000)
    .then(data => console.log('Got:', data))
    .catch(err => console.log('Race result:', err.message)); // Timeout!

In [ ]:
// Promise.any — first SUCCESS wins (ignores rejections)
async function demoAny() {
    try {
        const fastest = await Promise.any([
            fail(50, 'Server 1 down'),
            delay(200, 'Server 2 response'),
            delay(100, 'Server 3 response')
        ]);
        console.log('Promise.any — first success:', fastest);
    } catch (err) {
        console.log('All failed:', err.errors);
    }
}
demoAny();

---
## 6. Microtasks vs Macrotasks

This is a **favorite interview deep-dive** — understanding the task queue priority.

### Microtasks (drain completely between each macrotask):
- `process.nextTick()` (Node.js only, highest priority)
- `Promise.then / catch / finally`
- `queueMicrotask()`

### Macrotasks (one per event loop tick):
- `setTimeout` / `setInterval`
- `setImmediate` (Node.js only)
- I/O callbacks
- UI rendering (browser only)

In [ ]:
// INTERVIEW CLASSIC: Predict the output order

console.log('1');

setTimeout(() => console.log('2'), 0);

Promise.resolve().then(() => {
    console.log('3');
    Promise.resolve().then(() => console.log('4'));
});

Promise.resolve().then(() => console.log('5'));

console.log('6');

// Answer: 1, 6, 3, 5, 4, 2
// Explanation:
// - 1, 6: synchronous
// - 3, 5: microtasks (Promise.then) — both queued, run in order
// - 4: microtask queued DURING microtask processing — runs before macrotasks
// - 2: macrotask (setTimeout)

---
## 7. Common Async Patterns

In [ ]:
// Pattern 1: Sequential vs Parallel execution

const task = (name, ms) => new Promise(resolve => {
    setTimeout(() => {
        console.log(`  ${name} done`);
        resolve(name);
    }, ms);
});

// SEQUENTIAL — each waits for the previous (slower)
async function sequential() {
    console.time('sequential');
    await task('A', 100);
    await task('B', 100);
    await task('C', 100);
    console.timeEnd('sequential'); // ~300ms
}

// PARALLEL — all start at once (faster!)
async function parallel() {
    console.time('parallel');
    await Promise.all([
        task('A', 100),
        task('B', 100),
        task('C', 100)
    ]);
    console.timeEnd('parallel'); // ~100ms
}

sequential().then(() => parallel());

In [ ]:
// Pattern 2: Async iteration with for...of (sequential on purpose)
async function processItems(items) {
    const results = [];
    for (const item of items) {
        const result = await processOne(item); // intentionally sequential
        results.push(result);
    }
    return results;
}

async function processOne(item) {
    return new Promise(resolve =>
        setTimeout(() => resolve(item * 2), 50)
    );
}

processItems([1, 2, 3, 4, 5]).then(r => console.log('Sequential results:', r));

In [ ]:
// Pattern 3: Concurrency limiter (process N at a time)
async function withConcurrency(items, fn, concurrency = 3) {
    const results = [];
    for (let i = 0; i < items.length; i += concurrency) {
        const batch = items.slice(i, i + concurrency);
        const batchResults = await Promise.all(batch.map(fn));
        results.push(...batchResults);
    }
    return results;
}

const urls = [1, 2, 3, 4, 5, 6, 7, 8];
withConcurrency(urls, async (n) => {
    await new Promise(r => setTimeout(r, 100));
    return n * 10;
}, 3).then(r => console.log('Batch results:', r));

---
## 8. Error Handling in Async Code

### The golden rule:
> **Every Promise must have a `.catch()` or be inside a `try/catch` with `await`.** Unhandled rejections crash Node.js (v15+).

In [ ]:
// WRONG: forgotten error handling
// async function bad() {
//     const data = await fetchData(); // if this rejects → unhandled rejection!
// }

// CORRECT: always handle errors
async function good() {
    try {
        const data = await Promise.reject(new Error('API failed'));
    } catch (err) {
        console.log('Properly caught:', err.message);
    }
}

good();

In [ ]:
// COMMON MISTAKE: await inside forEach does NOT work as expected!

// WRONG — forEach doesn't wait for async callbacks
async function wrongWay() {
    const items = [1, 2, 3];
    items.forEach(async (item) => {
        await new Promise(r => setTimeout(r, 100));
        console.log('forEach item:', item);
    });
    console.log('forEach: "Done" prints BEFORE items!'); // This runs first!
}

// CORRECT — use for...of for sequential, Promise.all(map) for parallel
async function rightWay() {
    const items = [1, 2, 3];
    for (const item of items) {
        await new Promise(r => setTimeout(r, 100));
        console.log('for...of item:', item);
    }
    console.log('for...of: "Done" prints AFTER all items!');
}

wrongWay().then(() => setTimeout(rightWay, 500));

---
## 9. Interview Questions & Answers

### Q1: What's the difference between a callback, a Promise, and async/await?
**A:** Callbacks are functions passed to async operations (error-first pattern). Promises represent a future value with `.then()/.catch()` chaining. async/await is syntactic sugar over Promises that makes code look synchronous. They all handle async operations, just with increasing readability.

### Q2: What is callback hell and how do you solve it?
**A:** Callback hell is deeply nested callbacks forming a "pyramid of doom." Solutions: (1) Use Promises with chaining, (2) Use async/await, (3) Modularize into named functions, (4) Use libraries like `async.js`.

### Q3: Explain `Promise.all()` vs `Promise.allSettled()`.
**A:** `Promise.all()` rejects immediately if ANY promise rejects — use when all results are required. `Promise.allSettled()` waits for ALL to settle and never rejects — use when you want results regardless of individual failures.

### Q4: Can you use `await` without `async`?
**A:** In CommonJS modules, no — `await` must be inside an `async` function. In ESM with top-level await support (Node 14.8+), you can use `await` at the module top level.

### Q5: What happens if you forget to `await` a Promise?
**A:** The function continues without waiting for the result. The variable holds the Promise object, not the resolved value. Errors may become unhandled rejections. This is a common source of bugs.

### Q6: How would you implement a retry mechanism for an async operation?
**A:**
```javascript
async function retry(fn, retries = 3, delay = 1000) {
    for (let i = 0; i < retries; i++) {
        try {
            return await fn();
        } catch (err) {
            if (i === retries - 1) throw err;
            await new Promise(r => setTimeout(r, delay * (i + 1)));
        }
    }
}
```